# 00 · Setup & data

Sanity-check the environment, point the notebooks at a real BOLD run on the VM, and take a first look at the data.

**Run order:** `00_setup_and_data` → `01_interpolation_results` → `02_pipeline_comparison`.

All the heavy lifting lives in [`nb_utils.py`](nb_utils.py); the notebooks stay thin so they read well in a presentation.

> Run from the repo root with the env active, e.g. `source /srv/venvs/team4dbrain/setup_env.sh` first.

In [ ]:
%load_ext autoreload
%autoreload 2
import nb_utils as nb

nb.env_report()  # python / torch / cuda / nibabel

## Configuration — edit this once

`Config` holds every path the notebooks need. On the VM the defaults are usually right; you mainly set `data_dir` and (optionally) pin a specific `bold_file`.

In [ ]:
cfg = nb.Config(
    data_dir="/srv/fMRI-data",   # <- folder of *_bold.nii.gz runs on the VM
    bold_file=None,               # <- or pin one run, e.g. ".../sub-01_ses-00_task-ArchiSocial_dir-ap_bold.nii.gz"
    norm_mode="zscore",          # pretrained interp checkpoint = zscore, non-residual
    residual=False,
)
print("interp weights:", cfg.interp_weights, "exists:", cfg.interp_weights.is_file())
print("output dir:    ", cfg.out_dir)

## What runs are available?

In [ ]:
files = nb.list_bold_files(cfg.data_dir)
print(f"found {len(files)} BOLD run(s) under {cfg.data_dir}")
for f in files[:15]:
    print("  ", f.name)

In [ ]:
bold = cfg.resolve_bold()   # the run the demo notebooks will use
print("using:", bold)
nb.bold_info(bold)          # shape / voxel size / TR (header only)

## First look — a few axial slices at one timepoint

Just to confirm the run loads and looks like a brain before we model it.

In [ ]:
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt

img = nib.load(str(bold))
t0 = img.shape[3] // 2
vol = np.asarray(img.dataobj[..., t0], dtype=np.float32)
vmax = float(np.percentile(vol, 99.5))
Z = vol.shape[2]
zs = [int(Z * f) for f in (0.3, 0.45, 0.6, 0.75)]

fig, ax = plt.subplots(1, len(zs), figsize=(4 * len(zs), 4))
for a, z in zip(ax, zs):
    a.imshow(vol[:, :, z].T, origin="lower", cmap="gray", vmin=0, vmax=vmax)
    a.set_title(f"z={z}"); a.axis("off")
fig.suptitle(f"{bold.name}  ·  t={t0}")
plt.show()

All good — continue to **`01_interpolation_results.ipynb`** for the temporal-interpolation results, then **`02_pipeline_comparison.ipynb`** to compare full pipelines.